# General

For more informations, si the documentation *Documentary Strategy*.

# Import & Configs

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [2]:
print(PROJECT_ROOT)

/home/jeremy/Documents/dev/LLM_RAG/Medical_assistant


In [4]:
%load_ext autoreload
%autoreload 2

from src.retrieval.retriever import MedicalRetriever
from src.preprocessing.query_processing import clean_query

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
from src.utils.config import load_config

config = load_config()

config

{'models': {'embedding': 'sentence-transformers/all-MiniLM-L6-v2'},
 'retrieval': {'top_k': 3},
 'chunking': {'chunk_size': 500, 'overlap_sentences': 1}}

# Test Retriever Class

## Simple Retrieve

In [6]:
retriever = MedicalRetriever()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
results = retriever.retrieve(
    query="MRI diagnosis of glioblastoma",
    #top_k=3
)

In [8]:
for i, doc in enumerate(results["documents"][0]):

    print("\n" + "="*50)

    print(f"RESULT {i+1}")

    print(doc[:500])


RESULT 1
Impact on survival of glioblastoma patient's in relation to the imaging of the peri-surgical area: a multi-parametric diffusion MRI, perfusion MRI and [11C]MET PET study. OBJECTIVES: Glioblastoma (GBM) is an aggressive brain tumour with poor prognosis; recurrence near the surgical cavity is common despite multimodal-treatment. Conventional MRI lacks accuracy for early detection of recurrence and cannot reliably differentiate progression from pseudoprogression.

RESULT 2
Gliomas are the most common primary malignant brain tumors and are characterized by heterogeneous growth and complex biology, which complicate accurate diagnosis and management. While magnetic resonance imaging (MRI) remains the clinical standard, its limitations in delineating tumor margins and distinguishing recurrence from treatment-induced changes highlight the need for complementary molecular imaging.

RESULT 3
METHODS: Clinical 3-Tesla brain MRI images from 644 patients with pathologically confirmed adul

## Retrieve With Metadata

In [11]:
results = retriever.retrieve_with_metadata(

    "brain edema",

    top_k=2
)

In [12]:
results

[{'text': 'Peritumoral Edema in Neuro-oncology: Mechanisms, Imaging Biomarkers, and Emerging Therapeutic Methods. Peritumoral edema (PTE) occurs with nearly all intracranial tumors and their treatments. PTE increases mass effect and intracranial pressure. This often causes neurologic decline, worsens function, and complicates treatment planning. At times, emergency care is needed. This review examines current PTE findings and their relevance to neuroimaging and neuro-oncology management.',
  'metadata': {'pmid': '42163672',
   'year': 2026,
   'title': 'Peritumoral Edema in Neuro-oncology: Mechanisms, Imaging Biomarkers, and Emerging Therapeutic Methods'}},
 {'text': 'While tumor size and extent of resection (EOR) are recognized risk factors, the prognostic impact of peritumoral brain edema volume (PTBEV) is not fully established. This study investigated whether preoperative tumor volume (TV) and PTBEV independently predict postoperative ischemia and neurological morbidity. METHODS: We

## Retrieve With Year Filter

In [14]:
results = retriever.retrieve_recent(

    query="glioblastoma prognosis",

    top_k=3,

    min_year=2023
)

results

{'ids': [['42189415_0', '41890862_1', '42200192_1']],
 'embeddings': None,
 'documents': [['Peripheral hematological landscapes as biomarkers for detecting postoperative progression in glioblastoma multiforme: a multivariable risk scoring approach. PURPOSE: Glioblastoma multiforme (GBM) is the most common malignant tumor with poor prognosis despite standard treatment. While various hematological parameters are prognostic for GBM survival, their potential in disease monitoring remains underexplored.',
   'BACKGROUND: Glioblastomas (GBM) are highly aggressive, treatment-resistant brain tumors lacking clinically actionable, noninvasive prognostic biomarkers. Tumor response after standard-of-care chemoradiation (CRT) is difficult to interpret on imaging, and post-CRT MRI changes have not been well linked to molecular features or potential biomarkers.',
   'This study aimed to identify predictors of early treatment failure and their association with overall survival (OS). METHODS: We perfor

# Query Cleaning

In [16]:
clean_query(
    "Why MRI is not efficient for glioma?"
)

'mri not efficient for glioma'

# Retrieval Comparison

In [7]:
retriever = MedicalRetriever()

results = retriever.retrieve(
    query="MRI diagnosis of glioblastoma",
    #top_k=3
)

for i, doc in enumerate(results["documents"][0]):

    print("\n" + "="*50)

    print(f"RESULT {i+1}")

    print(doc[:500])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Original query: MRI diagnosis of glioblastoma
Processed query: mri diagnosis of glioblastoma

RESULT 1
Impact on survival of glioblastoma patient's in relation to the imaging of the peri-surgical area: a multi-parametric diffusion MRI, perfusion MRI and [11C]MET PET study. OBJECTIVES: Glioblastoma (GBM) is an aggressive brain tumour with poor prognosis; recurrence near the surgical cavity is common despite multimodal-treatment. Conventional MRI lacks accuracy for early detection of recurrence and cannot reliably differentiate progression from pseudoprogression.

RESULT 2
Gliomas are the most common primary malignant brain tumors and are characterized by heterogeneous growth and complex biology, which complicate accurate diagnosis and management. While magnetic resonance imaging (MRI) remains the clinical standard, its limitations in delineating tumor margins and distinguishing recurrence from treatment-induced changes highlight the need for complementary molecular imaging.

RESULT 3
ME

In [8]:
results = retriever.retrieve_recent(

    query="glioblastoma prognosis",

    top_k=3,

    min_year=2023
)

results

Original query: glioblastoma prognosis
Processed query: glioblastoma prognosis


{'ids': [['42189415_0', '41890862_1', '42200192_1']],
 'embeddings': None,
 'documents': [['Peripheral hematological landscapes as biomarkers for detecting postoperative progression in glioblastoma multiforme: a multivariable risk scoring approach. PURPOSE: Glioblastoma multiforme (GBM) is the most common malignant tumor with poor prognosis despite standard treatment. While various hematological parameters are prognostic for GBM survival, their potential in disease monitoring remains underexplored.',
   'BACKGROUND: Glioblastomas (GBM) are highly aggressive, treatment-resistant brain tumors lacking clinically actionable, noninvasive prognostic biomarkers. Tumor response after standard-of-care chemoradiation (CRT) is difficult to interpret on imaging, and post-CRT MRI changes have not been well linked to molecular features or potential biomarkers.',
   'This study aimed to identify predictors of early treatment failure and their association with overall survival (OS). METHODS: We perfor

# Analysis

## Retrieval Quality

The retrieval system, which is based on embeddings and ChromaDB, generally returns documents that are consistent with the entered queries.

Tests conducted on several medical queries demonstrate that the retrieved chunks are largely relevant to the search intent.

## Document noise

The observed noise level is low.

However, some edge cases remain:

- very general documents
- partially relevant chunks
- occasional retrieval of less specific documents

Future improvements could be made through:

- more advanced query cleaning
- query rewriting
- reranking

## Identified Limitations 
### Document Diversity

Currently, the system does not guarantee that chunks come from different documents.

In some cases, multiple chunks from the same article may dominate the context sent to the LLM.

A strategy to diversify the results may be added at a later date.

### Recency of Knowledge:

The system already utilizes temporal metadata.

A future improvement could involve integrating a hybrid score.

> Semantic relevance + recency of publication

This would prioritize the most recent publications.

## Query preprocessing:

The current preprocessing primarily involves the following:

- normalization
- stopword removal

This approach remains straightforward.

Future improvements could include:

- lemmatization
- extraction of medical entities
- automatic query reformulation using an LLM

## Conclusion:

The retrieval system provides a solid foundation for the initial project version.

The main objectives of the sprint have been achieved:

- cleaned corpus
- consistent chunking
- generated embeddings
- operational ChromaDB vectorization
- functional semantic search